In [ ]:
# ============================================================
# COMPLETE MEMORY / CACHE / VARIABLE CLEANUP
# ============================================================

import os, gc, time, psutil
def get_user_variables():
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None: return {}
        excluded = {"In", "Out", "get_ipython", "exit", "quit"}
        return {name: value for name, value in ip.user_ns.items() if not name.startswith("_") and name not in excluded}
    except Exception:
        return {}
def get_ram_usage():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2
def print_variable_information():
    variables = get_user_variables()
    print(f"Variable count  : {len(variables)}")
    if variables: print("Variables       : " + ", ".join(sorted(variables.keys())))
def clear_matplotlib_cache():
    try:
        import matplotlib.pyplot as plt
        print("\n[1] Closing Matplotlib figures...")
        plt.close("all")
        print("    Matplotlib figures closed.")
    except ImportError:
        print("\n[1] Matplotlib not installed.")
    except Exception as e:
        print(f"\n[1] Matplotlib warning: {e}")
def clear_tensorflow_memory():
    try:
        import tensorflow as tf
        print("\n[2] Clearing TensorFlow / Keras...")
        tf.keras.backend.clear_session()
        print("    Keras session cleared.")
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            for i in range(len(gpus)):
                try:
                    info = tf.config.experimental.get_memory_info(f"GPU:{i}")
                    print(f"    GPU:{i} current: {info['current']/1024**2:.2f} MB")
                    print(f"    GPU:{i} peak   : {info['peak']/1024**2:.2f} MB")
                except Exception:
                    pass
                try: tf.config.experimental.reset_memory_stats(f"GPU:{i}")
                except Exception: pass
        print("    TensorFlow cleanup completed.")
    except ImportError:
        print("\n[2] TensorFlow not installed.")
    except Exception as e:
        print(f"\n[2] TensorFlow warning: {e}")
def force_garbage_collection(cycles=3):
    print("\n[3] Running garbage collection...")
    total = sum(gc.collect() for _ in range(cycles))
    print(f"    Garbage-collected objects: {total}")
def delete_model_variables():
    print("\n[4] Deleting user/model/data variables...")
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None:
            print("    Not running inside Jupyter.")
            return 0, []
        protected = {"os", "gc", "time", "psutil", "get_user_variables", "get_ram_usage", "print_variable_information", "clear_matplotlib_cache", "clear_tensorflow_memory", "force_garbage_collection", "delete_model_variables", "clear_cache_and_memory", "In", "Out", "get_ipython", "exit", "quit"}
        names = [name for name in list(ip.user_ns.keys()) if not name.startswith("_") and name not in protected]
        print(f"    Variables selected for deletion: {len(names)}")
        if names: print("    Deleting: " + ", ".join(sorted(names)))
        deleted = 0
        for name in names:
            try:
                del ip.user_ns[name]
                deleted += 1
            except Exception as e:
                print(f"    Could not delete {name}: {e}")
        print(f"    Successfully deleted: {deleted}")
        return deleted, names
    except Exception as e:
        print(f"    Variable cleanup warning: {e}")
        return 0, []
def clear_cache_and_memory():
    print("\n" + "=" * 70)
    print("COMPLETE MEMORY / CACHE / VARIABLE CLEANUP")
    print("=" * 70)
    before_ram = get_ram_usage()
    before_variables = get_user_variables()
    before_count = len(before_variables)
    print("\nBEFORE CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {before_ram:.2f} MB")
    print(f"Variable count  : {before_count}")
    if before_variables: print("Variables       : " + ", ".join(sorted(before_variables.keys())))
    clear_matplotlib_cache()
    clear_tensorflow_memory()
    force_garbage_collection()
    deleted_count, deleted_names = delete_model_variables()
    print("\n[5] Final garbage collection...")
    total = sum(gc.collect() for _ in range(5))
    print(f"    Garbage-collected objects: {total}")
    time.sleep(1)
    after_ram = get_ram_usage()
    after_variables = get_user_variables()
    after_count = len(after_variables)
    print("\nAFTER CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {after_ram:.2f} MB")
    print(f"Variable count  : {after_count}")
    if after_variables: print("Remaining       : " + ", ".join(sorted(after_variables.keys())))
    ram_difference = before_ram - after_ram
    print("\n" + "=" * 70)
    print("CLEANUP SUMMARY")
    print("=" * 70)
    print(f"Variables before : {before_count}")
    print(f"Variables after  : {after_count}")
    print(f"Variables deleted: {deleted_count}")
    print("-" * 70)
    print(f"RAM before       : {before_ram:.2f} MB")
    print(f"RAM after        : {after_ram:.2f} MB")
    if ram_difference >= 0: print(f"RAM released     : {ram_difference:.2f} MB")
    else: print(f"RAM difference   : +{abs(ram_difference):.2f} MB")
    print("=" * 70)
    print("CLEANUP COMPLETED")
    print("=" * 70)
clear_cache_and_memory()


***
<a name='import Packages'>
    
# 1 <span style='color:blue'>|</span> GCN_KNN_Cosine

In [ ]:
# ============================================================
# GCN_KNN_Cosine
# Pure GCN on RAW PIXELS — kNN Cosine Graph
# ============================================================

import os, glob, time
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from scipy.sparse.csgraph import connected_components
import tensorflow as tf
from tensorflow.keras import layers, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# ============================================================
# STEP 0 — PATHS AND PARAMETERS
# ============================================================

data_dir = r"D:/GCN/Brain_Tumor/four_class"
IMG_SIZE_RAW = (112, 112)
SEED = 123
K = 12
LR_GCN = 5e-3
WD = 5e-4
EPOCHS_GCN = 250
HIDDEN = 64
DROPOUT = 0.3
LABEL_SMOOTH = 0.05
TEST_SIZE = 0.10
VAL_SPLIT = 2 / 9
USE_CLASS_BALANCING = True
TSNE_MAX_N = 2500

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# STEP 1 — INDEX IMAGES AND LABELS
# ============================================================

class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c: i for i, c in enumerate(class_names)}
paths = []
labels_int = []

for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels_int.append(class_to_idx[c])

paths = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)

if len(paths) == 0: raise ValueError(f"No images found in {data_dir}")
if num_classes < 2: raise ValueError("At least two classes are required.")

print(f"[STEP 1] Found {len(paths)} images across {num_classes} classes:")
print(class_names)

# ============================================================
# STEP 1A — STRATIFIED 70/20/10 SPLIT
# ============================================================

all_idx = np.arange(len(paths))
idx_trainval, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_trainval, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_trainval])

N = len(paths)
mask_tr = np.zeros(N, dtype=bool)
mask_va = np.zeros(N, dtype=bool)
mask_te = np.zeros(N, dtype=bool)
mask_tr[idx_tr] = True
mask_va[idx_va] = True
mask_te[idx_te] = True

if len(idx_tr) + len(idx_va) + len(idx_te) != N: raise ValueError("Dataset split size mismatch.")
if np.any(mask_tr & mask_va) or np.any(mask_tr & mask_te) or np.any(mask_va & mask_te): raise ValueError("Dataset split overlap detected.")

print("\n=== Dataset Split ===")
print(f"Training   : {len(idx_tr)} ({100 * len(idx_tr) / N:.2f}%)")
print(f"Validation : {len(idx_va)} ({100 * len(idx_va) / N:.2f}%)")
print(f"Testing    : {len(idx_te)} ({100 * len(idx_te) / N:.2f}%)")

print("\n=== Class Distribution ===")
for i, cname in enumerate(class_names):
    n_train = np.sum(labels_int[idx_tr] == i)
    n_val = np.sum(labels_int[idx_va] == i)
    n_test = np.sum(labels_int[idx_te] == i)
    print(f"{cname}: Train={n_train}, Validation={n_val}, Test={n_test}")

# ============================================================
# STEP 2 — LOAD RAW PIXELS
# ============================================================

def load_flat(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE_RAW, method=tf.image.ResizeMethod.BILINEAR)
    img.set_shape([IMG_SIZE_RAW[0], IMG_SIZE_RAW[1], 3])
    return tf.reshape(img, [-1])

def extract_raw_features(paths_np, batch=64):
    feats = []
    for i in range(0, len(paths_np), batch):
        pb = paths_np[i:i + batch]
        xb = tf.stack([load_flat(p) for p in pb], axis=0).numpy()
        feats.append(xb)
    return np.vstack(feats).astype(np.float32)

print("\n[STEP 2] Extracting raw-pixel features...")
t0 = time.time()
X_raw = extract_raw_features(paths)
t1 = time.time()

print(f"[TIME] Feature extraction: {(t1 - t0) / 60:.2f} minutes")

N, F = X_raw.shape

print("\n=== Raw Feature Information ===")
print(f"Nodes N = {N}")
print(f"Feature dimension F = {F}")
print(f"Image dimensions = {IMG_SIZE_RAW[0]} × {IMG_SIZE_RAW[1]} × 3")

y_onehot = to_categorical(labels_int, num_classes=num_classes).astype(np.float32)

# ============================================================
# STEP 2A — STANDARDIZATION
# ============================================================

scaler = StandardScaler(with_mean=True, with_std=True)
scaler.fit(X_raw[idx_tr])
X_std = scaler.transform(X_raw).astype(np.float32)

print("[STEP 2A] Raw-pixel features standardized using training-set statistics.")

# ============================================================
# STEP 3 — BUILD kNN COSINE GRAPH
# ============================================================

print("\n[STEP 3] Building kNN Cosine Graph...")
t2 = time.time()

nbrs = NearestNeighbors(n_neighbors=K + 1, metric="cosine", n_jobs=-1)
nbrs.fit(X_std)
dist, knn_idx = nbrs.kneighbors(X_std, return_distance=True)

rows = []
cols = []
data = []

for i in range(N):
    for j, d in zip(knn_idx[i], dist[i]):
        if i == j: continue
        similarity = 1.0 - float(d)
        if similarity <= 0: continue
        rows.append(i)
        cols.append(j)
        data.append(similarity)

A_dir = sp.coo_matrix((np.asarray(data, dtype=np.float32), (np.asarray(rows, dtype=np.int32), np.asarray(cols, dtype=np.int32))), shape=(N, N), dtype=np.float32)
A_knn = A_dir.minimum(A_dir.T)
A_knn = A_knn.tolil()
A_knn.setdiag(1.0)
A_knn = A_knn.tocsr()
A_norm = gcn_filter(A_knn).astype(np.float32)

t3 = time.time()

print(f"[TIME] Graph construction + normalization: {(t3 - t2) / 60:.2f} minutes")

# ============================================================
# STEP 3A — GRAPH STATISTICS
# ============================================================

A_no_self = A_knn.copy().tocsr()
A_no_self.setdiag(0)
A_no_self.eliminate_zeros()

degree_counts = np.diff(A_no_self.indptr)
num_edges = A_no_self.nnz // 2
avg_degree = degree_counts.mean()
min_degree = degree_counts.min()
max_degree = degree_counts.max()
graph_density = (2.0 * num_edges) / (N * (N - 1)) if N > 1 else 0.0
adjacency_nnz = A_knn.nnz
n_components, component_labels = connected_components(A_no_self, directed=False, return_labels=True)
component_sizes = np.bincount(component_labels)
largest_cc = component_sizes.max()

print("\n=== GCN_KNN_Cosine Graph Statistics ===")
print(f"Nodes          : {N}")
print(f"Edges          : {num_edges}")
print(f"K              : {K}")
print(f"Average degree : {avg_degree:.2f}")
print(f"Minimum degree : {min_degree}")
print(f"Maximum degree : {max_degree}")
print(f"Density        : {graph_density:.6f}")
print(f"Components     : {n_components}")
print(f"LargestCC      : {largest_cc}")
print(f"nnz(A)         : {adjacency_nnz}")

# ============================================================
# STEP 3B — t-SNE OF RAW-PIXEL FEATURES
# ============================================================

print("\n[VIS] Running t-SNE on raw features...")

if N > TSNE_MAX_N:
    sel = np.random.RandomState(SEED).choice(N, TSNE_MAX_N, replace=False)
    X_for_tsne = X_std[sel]
    y_for_tsne = labels_int[sel]
else:
    X_for_tsne = X_std
    y_for_tsne = labels_int

tsne = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=SEED)
X_2d = tsne.fit_transform(X_for_tsne)

plt.figure(figsize=(7, 6))
for i, cname in enumerate(class_names):
    m = y_for_tsne == i
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=8, alpha=0.7, label=cname)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE of Raw-Pixel Features")
plt.legend(markerscale=3, frameon=False)
plt.tight_layout()
plt.show()

# ============================================================
# STEP 3C — DEGREE DISTRIBUTION
# ============================================================

plt.figure(figsize=(7, 4))
plt.hist(degree_counts, bins=range(int(degree_counts.min()), int(degree_counts.max()) + 2), edgecolor="black", alpha=0.85)
plt.xlabel("Node Degree")
plt.ylabel("Number of Nodes")
plt.title("Degree Distribution of kNN Cosine Graph")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 3D — ADJACENCY MATRIX SNAPSHOT
# ============================================================

subN = min(150, N)
A_small = A_knn[:subN, :subN].toarray()
plt.figure(figsize=(5.5, 5))
plt.imshow(A_small, cmap="Greys", interpolation="nearest", aspect="auto")
plt.xlabel("Node Index")
plt.ylabel("Node Index")
plt.title(f"kNN Cosine Adjacency Snapshot ({subN}×{subN})")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 4 — CLASS BALANCING
# ============================================================

train_counts = np.bincount(labels_int[idx_tr], minlength=num_classes).astype(np.float32)
class_w = np.ones(num_classes, dtype=np.float32)
sample_w_train = mask_tr.astype(np.float32)

if USE_CLASS_BALANCING:
    class_w = train_counts.sum() / (num_classes * np.maximum(train_counts, 1.0))
    sample_w_train = sample_w_train * class_w[labels_int]

print("\n=== Training Class Weights ===")
for i, cname in enumerate(class_names):
    print(f"{cname}: Count={int(train_counts[i])}, Weight={class_w[i]:.4f}")

# ============================================================
# STEP 5 — PURE GCN MODEL
# ============================================================

X_in = Input(shape=(X_std.shape[1],), name="X_in")
A_in = Input(shape=(N,), sparse=True, name="A_in")
h1 = GCNConv(HIDDEN, activation=None, kernel_regularizer=regularizers.l2(WD), name="GCN_1")([X_in, A_in])
h1 = layers.BatchNormalization(name="BatchNorm")(h1)
h1 = layers.Activation("relu", name="ReLU")(h1)
h1 = layers.Dropout(DROPOUT, name="Dropout")(h1)
res = layers.Dense(HIDDEN, use_bias=False, name="Residual_Projection")(X_in)
h1 = layers.Add(name="Residual_Add")([h1, res])
out = GCNConv(num_classes, activation="softmax", name="GCN_Output")([h1, A_in])
gcn = Model(inputs=[X_in, A_in], outputs=out, name="GCN_KNN_Cosine")

gcn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR_GCN), loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH), weighted_metrics=["accuracy"])
gcn.summary()

# ============================================================
# STEP 5A — CALLBACKS
# ============================================================

ckpt_gcn = "best_GCN_KNN_Cosine.keras"
callbacks_gcn = [EarlyStopping(monitor="val_loss", patience=20, mode="min", restore_best_weights=True, verbose=1), ModelCheckpoint(ckpt_gcn, monitor="val_loss", mode="min", save_best_only=True, verbose=1)]

# ============================================================
# STEP 5B — TRAIN MODEL
# ============================================================

print("\n[STEP 5] Training GCN_KNN_Cosine...")
t4 = time.time()

hist = gcn.fit(x=[X_std, A_norm], y=y_onehot, sample_weight=sample_w_train, batch_size=N, epochs=EPOCHS_GCN, shuffle=False, verbose=1, validation_data=([X_std, A_norm], y_onehot, mask_va.astype(np.float32)), callbacks=callbacks_gcn)

t5 = time.time()
training_time = t5 - t4

print(f"\n[TIME] GCN training: {training_time / 60:.2f} minutes")

# ============================================================
# STEP 5C — TRAINING AND VALIDATION CURVES
# ============================================================

plt.figure(figsize=(7, 5))
plt.plot(hist.history["accuracy"], label="Training Accuracy")
plt.plot(hist.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("GCN kNN Cosine: Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(hist.history["loss"], label="Training Loss")
plt.plot(hist.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GCN kNN Cosine: Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# STEP 6 — PREDICTIONS
# ============================================================

print("\n[STEP 6] Generating predictions...")
t6 = time.time()
y_prob = gcn.predict([X_std, A_norm], batch_size=N, verbose=0)
t7 = time.time()

inference_time = t7 - t6
y_pred = np.argmax(y_prob, axis=1)

print(f"[TIME] Full-graph inference: {inference_time:.4f} seconds")

# ============================================================
# STEP 6A — EVALUATION FUNCTION
# ============================================================

def evaluate_mask(mask_name, mask_bool):
    loss, acc = gcn.evaluate([X_std, A_norm], y_onehot, sample_weight=mask_bool.astype(np.float32), batch_size=N, verbose=0)

    print(f"\n{'=' * 60}")
    print(f"{mask_name} RESULTS")
    print(f"{'=' * 60}")
    print(f"Loss     : {loss:.4f}")
    print(f"Accuracy : {acc:.4f}")

    y_true_mask = labels_int[mask_bool]
    y_pred_mask = y_pred[mask_bool]
    y_prob_mask = y_prob[mask_bool]

    print(f"\n[{mask_name}] Classification Report:")
    print(classification_report(y_true_mask, y_pred_mask, labels=np.arange(num_classes), target_names=class_names, digits=4, zero_division=0))

    cm = confusion_matrix(y_true_mask, y_pred_mask, labels=np.arange(num_classes))
    print(f"[{mask_name}] Confusion Matrix:")
    print(cm)

    plt.figure(figsize=(6.5, 6))
    plt.imshow(cm, cmap="Blues")
    plt.colorbar()
    plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
    plt.yticks(range(num_classes), class_names)
    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.title(f"GCN kNN Cosine Confusion Matrix ({mask_name})")
    threshold = cm.max() / 2.0 if cm.size > 0 else 0

    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, cm[i, j], ha="center", va="center", color=("white" if cm[i, j] > threshold else "black"))

    plt.tight_layout()
    plt.show()

    try:
        y_bin = label_binarize(y_true_mask, classes=np.arange(num_classes))
        if num_classes == 2: y_bin = np.column_stack([1 - y_bin[:, 0], y_bin[:, 0]])

        fpr = {}
        tpr = {}
        roc_auc = {}

        for i in range(num_classes):
            if len(np.unique(y_bin[:, i])) < 2: continue
            fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_prob_mask[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_prob_mask.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
        valid_classes = [i for i in range(num_classes) if i in fpr]

        if len(valid_classes) > 0:
            all_fpr = np.unique(np.concatenate([fpr[i] for i in valid_classes]))
            mean_tpr = np.zeros_like(all_fpr)
            for i in valid_classes: mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
            mean_tpr /= len(valid_classes)
            fpr["macro"] = all_fpr
            tpr["macro"] = mean_tpr
            roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

        plt.figure(figsize=(8, 6))

        for i in valid_classes:
            plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.3f})")

        plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"Micro-average (AUC={roc_auc['micro']:.3f})")

        if "macro" in fpr:
            plt.plot(fpr["macro"], tpr["macro"], linestyle=":", label=f"Macro-average (AUC={roc_auc['macro']:.3f})")

        plt.plot([0, 1], [0, 1], "k--")
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"GCN kNN Cosine ROC Curves ({mask_name})")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.show()

        print(f"\n[{mask_name}] AUC:")
        for i in valid_classes: print(f"{class_names[i]}: {roc_auc[i]:.4f}")
        print(f"Micro-average: {roc_auc['micro']:.4f}")
        if "macro" in roc_auc: print(f"Macro-average: {roc_auc['macro']:.4f}")

    except Exception as e:
        print(f"[ROC] Skipped: {e}")

# ============================================================
# STEP 6B — VALIDATION AND TEST RESULTS
# ============================================================

evaluate_mask("VALIDATION", mask_va)
evaluate_mask("TEST", mask_te)

# ============================================================
# FINAL COMPUTATIONAL INFORMATION — GCN_KNN_Cosine
# ============================================================

print("\n============================================================")
print("GCN_KNN_Cosine FINAL SUMMARY")
print("============================================================")
print(f"Model name           : {gcn.name}")
print(f"Input image size     : {IMG_SIZE_RAW[0]} × {IMG_SIZE_RAW[1]} × 3")
print(f"Raw feature dim      : {F}")
print(f"Number of nodes      : {N}")
print(f"Training nodes       : {len(idx_tr)}")
print(f"Validation nodes     : {len(idx_va)}")
print(f"Test nodes           : {len(idx_te)}")
print("Dataset split        : 70% / 20% / 10%")
print(f"kNN K                : {K}")
print("Neighbour metric     : Cosine")
print("Graph type           : Mutual kNN")
print("Edge weighting       : Cosine similarity")
print(f"Graph edges          : {num_edges}")
print(f"Average degree       : {avg_degree:.2f}")
print(f"Minimum degree       : {min_degree}")
print(f"Maximum degree       : {max_degree}")
print(f"Graph density        : {graph_density:.6f}")
print(f"Graph components     : {n_components}")
print(f"Largest component    : {largest_cc}")
print(f"Adjacency nnz        : {adjacency_nnz}")
print(f"Class balancing      : {USE_CLASS_BALANCING}")
print(f"GCN hidden units     : {HIDDEN}")
print(f"Dropout              : {DROPOUT}")
print(f"Learning rate        : {LR_GCN}")
print(f"Weight decay         : {WD}")
print(f"Label smoothing      : {LABEL_SMOOTH}")
print(f"Epochs               : {EPOCHS_GCN}")
print(f"Trainable parameters : {gcn.count_params():,}")
print(f"Training time        : {training_time:.2f} seconds")
print(f"Training time        : {training_time / 60:.2f} minutes")
print(f"Inference time       : {inference_time:.4f} seconds")
print("============================================================")


***
<a name='import Packages'>
    
# 0 <span style='color:blue'>|</span> COMPLETE MEMORY / CACHE / VARIABLE CLEANUP

In [ ]:
import os, gc, time, psutil

def get_user_variables():
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None: return {}
        excluded = {"In", "Out", "get_ipython", "exit", "quit"}
        return {name: value for name, value in ip.user_ns.items() if not name.startswith("_") and name not in excluded}
    except Exception:
        return {}
def get_ram_usage():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2
def print_variable_information():
    variables = get_user_variables()
    print(f"Variable count  : {len(variables)}")
    if variables: print("Variables       : " + ", ".join(sorted(variables.keys())))
def clear_matplotlib_cache():
    try:
        import matplotlib.pyplot as plt
        print("\n[1] Closing Matplotlib figures...")
        plt.close("all")
        print("    Matplotlib figures closed.")
    except ImportError:
        print("\n[1] Matplotlib not installed.")
    except Exception as e:
        print(f"\n[1] Matplotlib warning: {e}")
def clear_tensorflow_memory():
    try:
        import tensorflow as tf
        print("\n[2] Clearing TensorFlow / Keras...")
        tf.keras.backend.clear_session()
        print("    Keras session cleared.")
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            for i in range(len(gpus)):
                try:
                    info = tf.config.experimental.get_memory_info(f"GPU:{i}")
                    print(f"    GPU:{i} current: {info['current']/1024**2:.2f} MB")
                    print(f"    GPU:{i} peak   : {info['peak']/1024**2:.2f} MB")
                except Exception:
                    pass
                try: tf.config.experimental.reset_memory_stats(f"GPU:{i}")
                except Exception: pass
        print("    TensorFlow cleanup completed.")
    except ImportError:
        print("\n[2] TensorFlow not installed.")
    except Exception as e:
        print(f"\n[2] TensorFlow warning: {e}")
def force_garbage_collection(cycles=3):
    print("\n[3] Running garbage collection...")
    total = sum(gc.collect() for _ in range(cycles))
    print(f"    Garbage-collected objects: {total}")
def delete_model_variables():
    print("\n[4] Deleting user/model/data variables...")
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None:
            print("    Not running inside Jupyter.")
            return 0, []
        protected = {"os", "gc", "time", "psutil", "get_user_variables", "get_ram_usage", "print_variable_information", "clear_matplotlib_cache", "clear_tensorflow_memory", "force_garbage_collection", "delete_model_variables", "clear_cache_and_memory", "In", "Out", "get_ipython", "exit", "quit"}
        names = [name for name in list(ip.user_ns.keys()) if not name.startswith("_") and name not in protected]
        print(f"    Variables selected for deletion: {len(names)}")
        if names: print("    Deleting: " + ", ".join(sorted(names)))
        deleted = 0
        for name in names:
            try:
                del ip.user_ns[name]
                deleted += 1
            except Exception as e:
                print(f"    Could not delete {name}: {e}")
        print(f"    Successfully deleted: {deleted}")
        return deleted, names
    except Exception as e:
        print(f"    Variable cleanup warning: {e}")
        return 0, []
def clear_cache_and_memory():
    print("\n" + "=" * 70)
    print("COMPLETE MEMORY / CACHE / VARIABLE CLEANUP")
    print("=" * 70)
    before_ram = get_ram_usage()
    before_variables = get_user_variables()
    before_count = len(before_variables)
    print("\nBEFORE CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {before_ram:.2f} MB")
    print(f"Variable count  : {before_count}")
    if before_variables: print("Variables       : " + ", ".join(sorted(before_variables.keys())))
    clear_matplotlib_cache()
    clear_tensorflow_memory()
    force_garbage_collection()
    deleted_count, deleted_names = delete_model_variables()
    print("\n[5] Final garbage collection...")
    total = sum(gc.collect() for _ in range(5))
    print(f"    Garbage-collected objects: {total}")
    time.sleep(1)
    after_ram = get_ram_usage()
    after_variables = get_user_variables()
    after_count = len(after_variables)
    print("\nAFTER CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {after_ram:.2f} MB")
    print(f"Variable count  : {after_count}")
    if after_variables: print("Remaining       : " + ", ".join(sorted(after_variables.keys())))
    ram_difference = before_ram - after_ram
    print("\n" + "=" * 70)
    print("CLEANUP SUMMARY")
    print("=" * 70)
    print(f"Variables before : {before_count}")
    print(f"Variables after  : {after_count}")
    print(f"Variables deleted: {deleted_count}")
    print("-" * 70)
    print(f"RAM before       : {before_ram:.2f} MB")
    print(f"RAM after        : {after_ram:.2f} MB")
    if ram_difference >= 0: print(f"RAM released     : {ram_difference:.2f} MB")
    else: print(f"RAM difference   : +{abs(ram_difference):.2f} MB")
    print("=" * 70)
    print("CLEANUP COMPLETED")
    print("=" * 70)
clear_cache_and_memory()


***
<a name='import Packages'>
    
# 1 <span style='color:blue'>|</span> K-fold Hybrid based KNN graph building model

In [ ]:
# ============================================================
# COMPLETE MEMORY / CACHE / VARIABLE CLEANUP
# ============================================================

import os, gc, time, psutil
def get_user_variables():
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None: return {}
        excluded = {"In", "Out", "get_ipython", "exit", "quit"}
        return {name: value for name, value in ip.user_ns.items() if not name.startswith("_") and name not in excluded}
    except Exception:
        return {}
def get_ram_usage():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2
def print_variable_information():
    variables = get_user_variables()
    print(f"Variable count  : {len(variables)}")
    if variables: print("Variables       : " + ", ".join(sorted(variables.keys())))
def clear_matplotlib_cache():
    try:
        import matplotlib.pyplot as plt
        print("\n[1] Closing Matplotlib figures...")
        plt.close("all")
        print("    Matplotlib figures closed.")
    except ImportError:
        print("\n[1] Matplotlib not installed.")
    except Exception as e:
        print(f"\n[1] Matplotlib warning: {e}")
def clear_tensorflow_memory():
    try:
        import tensorflow as tf
        print("\n[2] Clearing TensorFlow / Keras...")
        tf.keras.backend.clear_session()
        print("    Keras session cleared.")
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            for i in range(len(gpus)):
                try:
                    info = tf.config.experimental.get_memory_info(f"GPU:{i}")
                    print(f"    GPU:{i} current: {info['current']/1024**2:.2f} MB")
                    print(f"    GPU:{i} peak   : {info['peak']/1024**2:.2f} MB")
                except Exception:
                    pass
                try: tf.config.experimental.reset_memory_stats(f"GPU:{i}")
                except Exception: pass
        print("    TensorFlow cleanup completed.")
    except ImportError:
        print("\n[2] TensorFlow not installed.")
    except Exception as e:
        print(f"\n[2] TensorFlow warning: {e}")
def force_garbage_collection(cycles=3):
    print("\n[3] Running garbage collection...")
    total = sum(gc.collect() for _ in range(cycles))
    print(f"    Garbage-collected objects: {total}")
def delete_model_variables():
    print("\n[4] Deleting user/model/data variables...")
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None:
            print("    Not running inside Jupyter.")
            return 0, []
        protected = {"os", "gc", "time", "psutil", "get_user_variables", "get_ram_usage", "print_variable_information", "clear_matplotlib_cache", "clear_tensorflow_memory", "force_garbage_collection", "delete_model_variables", "clear_cache_and_memory", "In", "Out", "get_ipython", "exit", "quit"}
        names = [name for name in list(ip.user_ns.keys()) if not name.startswith("_") and name not in protected]
        print(f"    Variables selected for deletion: {len(names)}")
        if names: print("    Deleting: " + ", ".join(sorted(names)))
        deleted = 0
        for name in names:
            try:
                del ip.user_ns[name]
                deleted += 1
            except Exception as e:
                print(f"    Could not delete {name}: {e}")
        print(f"    Successfully deleted: {deleted}")
        return deleted, names
    except Exception as e:
        print(f"    Variable cleanup warning: {e}")
        return 0, []
def clear_cache_and_memory():
    print("\n" + "=" * 70)
    print("COMPLETE MEMORY / CACHE / VARIABLE CLEANUP")
    print("=" * 70)
    before_ram = get_ram_usage()
    before_variables = get_user_variables()
    before_count = len(before_variables)
    print("\nBEFORE CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {before_ram:.2f} MB")
    print(f"Variable count  : {before_count}")
    if before_variables: print("Variables       : " + ", ".join(sorted(before_variables.keys())))
    clear_matplotlib_cache()
    clear_tensorflow_memory()
    force_garbage_collection()
    deleted_count, deleted_names = delete_model_variables()
    print("\n[5] Final garbage collection...")
    total = sum(gc.collect() for _ in range(5))
    print(f"    Garbage-collected objects: {total}")
    time.sleep(1)
    after_ram = get_ram_usage()
    after_variables = get_user_variables()
    after_count = len(after_variables)
    print("\nAFTER CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {after_ram:.2f} MB")
    print(f"Variable count  : {after_count}")
    if after_variables: print("Remaining       : " + ", ".join(sorted(after_variables.keys())))
    ram_difference = before_ram - after_ram
    print("\n" + "=" * 70)
    print("CLEANUP SUMMARY")
    print("=" * 70)
    print(f"Variables before : {before_count}")
    print(f"Variables after  : {after_count}")
    print(f"Variables deleted: {deleted_count}")
    print("-" * 70)
    print(f"RAM before       : {before_ram:.2f} MB")
    print(f"RAM after        : {after_ram:.2f} MB")
    if ram_difference >= 0: print(f"RAM released     : {ram_difference:.2f} MB")
    else: print(f"RAM difference   : +{abs(ram_difference):.2f} MB")
    print("=" * 70)
    print("CLEANUP COMPLETED")
    print("=" * 70)
clear_cache_and_memory()


***
<a name='import Packages'>
    
# 1 <span style='color:blue'>|</span> K-fold Hybrid based KNN graph building model

In [ ]:
# ============================================================
# GCN_Epsilon_Cosine
# Pure GCN on RAW PIXELS — ε-Graph Using Cosine Similarity
# ============================================================

import os, glob, time
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.neighbors import radius_neighbors_graph
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from scipy.sparse.csgraph import connected_components
import tensorflow as tf
from tensorflow.keras import layers, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# ============================================================
# STEP 0 — PATHS AND PARAMETERS
# ============================================================

data_dir = r"D:/GCN/Brain_Tumor/four_class"
IMG_SIZE_RAW = (112, 112)
SEED = 123
EPS_COS = 0.50
LR_GCN = 5e-3
WD = 5e-4
EPOCHS_GCN = 250
HIDDEN = 64
DROPOUT = 0.3
LABEL_SMOOTH = 0.05
TEST_SIZE = 0.10
VAL_SPLIT = 2 / 9
USE_CLASS_BALANCING = True
TSNE_MAX_N = 2500

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# STEP 1 — INDEX IMAGES AND LABELS
# ============================================================

class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c: i for i, c in enumerate(class_names)}
paths = []
labels_int = []

for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels_int.append(class_to_idx[c])

paths = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)

if len(paths) == 0: raise ValueError(f"No images found in {data_dir}")
if num_classes < 2: raise ValueError("At least two classes are required.")

print(f"[STEP 1] Found {len(paths)} images across {num_classes} classes:")
print(class_names)

# ============================================================
# STEP 1A — STRATIFIED 70/20/10 SPLIT
# ============================================================

all_idx = np.arange(len(paths))
idx_trainval, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_trainval, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_trainval])

N = len(paths)
mask_tr = np.zeros(N, dtype=bool)
mask_va = np.zeros(N, dtype=bool)
mask_te = np.zeros(N, dtype=bool)
mask_tr[idx_tr] = True
mask_va[idx_va] = True
mask_te[idx_te] = True

if len(idx_tr) + len(idx_va) + len(idx_te) != N: raise ValueError("Dataset split size mismatch.")
if np.any(mask_tr & mask_va) or np.any(mask_tr & mask_te) or np.any(mask_va & mask_te): raise ValueError("Dataset split overlap detected.")

print("\n=== Dataset Split ===")
print(f"Training   : {len(idx_tr)} ({100 * len(idx_tr) / N:.2f}%)")
print(f"Validation : {len(idx_va)} ({100 * len(idx_va) / N:.2f}%)")
print(f"Testing    : {len(idx_te)} ({100 * len(idx_te) / N:.2f}%)")

print("\n=== Class Distribution ===")
for i, cname in enumerate(class_names):
    n_train = np.sum(labels_int[idx_tr] == i)
    n_val = np.sum(labels_int[idx_va] == i)
    n_test = np.sum(labels_int[idx_te] == i)
    print(f"{cname}: Train={n_train}, Validation={n_val}, Test={n_test}")

# ============================================================
# STEP 2 — LOAD RAW PIXELS
# ============================================================

def load_flat(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE_RAW, method=tf.image.ResizeMethod.BILINEAR)
    img.set_shape([IMG_SIZE_RAW[0], IMG_SIZE_RAW[1], 3])
    return tf.reshape(img, [-1])

def extract_raw_features(paths_np, batch=64):
    feats = []
    for i in range(0, len(paths_np), batch):
        pb = paths_np[i:i + batch]
        xb = tf.stack([load_flat(p) for p in pb], axis=0).numpy()
        feats.append(xb)
    return np.vstack(feats).astype(np.float32)

print("\n[STEP 2] Extracting raw-pixel features...")
t0 = time.time()
X_raw = extract_raw_features(paths)
t1 = time.time()

print(f"[TIME] Feature extraction: {(t1 - t0) / 60:.2f} minutes")

N, F = X_raw.shape

print("\n=== Raw Feature Information ===")
print(f"Nodes N = {N}")
print(f"Feature dimension F = {F}")
print(f"Image dimensions = {IMG_SIZE_RAW[0]} × {IMG_SIZE_RAW[1]} × 3")

y_onehot = to_categorical(labels_int, num_classes=num_classes).astype(np.float32)

# ============================================================
# STEP 2A — STANDARDIZATION
# ============================================================

scaler = StandardScaler(with_mean=True, with_std=True)
scaler.fit(X_raw[idx_tr])
X_std = scaler.transform(X_raw).astype(np.float32)

print("[STEP 2A] Raw-pixel features standardized using training-set statistics.")

# ============================================================
# STEP 3 — BUILD ε-GRAPH USING COSINE SIMILARITY
# ============================================================

print("\n[STEP 3] Building ε-Graph using cosine similarity...")
t2 = time.time()

radius = 1.0 - EPS_COS
A_eps = radius_neighbors_graph(X_std, radius=radius, metric="cosine", mode="distance", include_self=False, n_jobs=-1).tocsr()
A_eps.data = (1.0 - A_eps.data).astype(np.float32)
A_eps.eliminate_zeros()
A_eps = A_eps.maximum(A_eps.T)
A_eps = A_eps.tolil()
A_eps.setdiag(1.0)
A_eps = A_eps.tocsr()
A_norm = gcn_filter(A_eps).astype(np.float32)

t3 = time.time()

print(f"[TIME] Graph construction + normalization: {(t3 - t2) / 60:.2f} minutes")

# ============================================================
# STEP 3A — GRAPH STATISTICS
# ============================================================

A_no_self = A_eps.copy().tocsr()
A_no_self.setdiag(0)
A_no_self.eliminate_zeros()

degree_counts = np.diff(A_no_self.indptr)
num_edges = A_no_self.nnz // 2
avg_degree = degree_counts.mean()
min_degree = degree_counts.min()
max_degree = degree_counts.max()
graph_density = (2.0 * num_edges) / (N * (N - 1)) if N > 1 else 0.0
adjacency_nnz = A_eps.nnz
n_components, component_labels = connected_components(A_no_self, directed=False, return_labels=True)
component_sizes = np.bincount(component_labels)
largest_cc = component_sizes.max()

print("\n=== GCN_Epsilon_Cosine Graph Statistics ===")
print(f"Nodes          : {N}")
print(f"Edges          : {num_edges}")
print(f"Epsilon cosine : {EPS_COS}")
print(f"Average degree : {avg_degree:.2f}")
print(f"Minimum degree : {min_degree}")
print(f"Maximum degree : {max_degree}")
print(f"Density        : {graph_density:.6f}")
print(f"Components     : {n_components}")
print(f"LargestCC      : {largest_cc}")
print(f"nnz(A)         : {adjacency_nnz}")

# ============================================================
# STEP 3B — t-SNE OF RAW-PIXEL FEATURES
# ============================================================

print("\n[VIS] Running t-SNE on raw features...")

if N > TSNE_MAX_N:
    sel = np.random.RandomState(SEED).choice(N, TSNE_MAX_N, replace=False)
    X_for_tsne = X_std[sel]
    y_for_tsne = labels_int[sel]
else:
    X_for_tsne = X_std
    y_for_tsne = labels_int

tsne = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=SEED)
X_2d = tsne.fit_transform(X_for_tsne)

plt.figure(figsize=(7, 6))
for i, cname in enumerate(class_names):
    m = y_for_tsne == i
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=8, alpha=0.7, label=cname)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE of Raw-Pixel Features")
plt.legend(markerscale=3, frameon=False)
plt.tight_layout()
plt.show()

# ============================================================
# STEP 3C — DEGREE DISTRIBUTION
# ============================================================

plt.figure(figsize=(7, 4))
plt.hist(degree_counts, bins=50, edgecolor="black", alpha=0.85)
plt.xlabel("Node Degree")
plt.ylabel("Number of Nodes")
plt.title("Degree Distribution of ε-Cosine Graph")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 3D — ADJACENCY MATRIX SNAPSHOT
# ============================================================

subN = min(150, N)
A_small = A_eps[:subN, :subN].toarray()
plt.figure(figsize=(5.5, 5))
plt.imshow(A_small, cmap="Greys", interpolation="nearest", aspect="auto")
plt.xlabel("Node Index")
plt.ylabel("Node Index")
plt.title(f"ε-Cosine Adjacency Snapshot ({subN}×{subN})")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 4 — CLASS BALANCING
# ============================================================

train_counts = np.bincount(labels_int[idx_tr], minlength=num_classes).astype(np.float32)
class_w = np.ones(num_classes, dtype=np.float32)
sample_w_train = mask_tr.astype(np.float32)

if USE_CLASS_BALANCING:
    class_w = train_counts.sum() / (num_classes * np.maximum(train_counts, 1.0))
    sample_w_train = sample_w_train * class_w[labels_int]

print("\n=== Training Class Weights ===")
for i, cname in enumerate(class_names):
    print(f"{cname}: Count={int(train_counts[i])}, Weight={class_w[i]:.4f}")

# ============================================================
# STEP 5 — PURE GCN MODEL
# ============================================================

X_in = Input(shape=(X_std.shape[1],), name="X_in")
A_in = Input(shape=(N,), sparse=True, name="A_in")
h1 = GCNConv(HIDDEN, activation=None, kernel_regularizer=regularizers.l2(WD), name="GCN_1")([X_in, A_in])
h1 = layers.BatchNormalization(name="BatchNorm")(h1)
h1 = layers.Activation("relu", name="ReLU")(h1)
h1 = layers.Dropout(DROPOUT, name="Dropout")(h1)
res = layers.Dense(HIDDEN, use_bias=False, name="Residual_Projection")(X_in)
h1 = layers.Add(name="Residual_Add")([h1, res])
out = GCNConv(num_classes, activation="softmax", name="GCN_Output")([h1, A_in])
gcn = Model(inputs=[X_in, A_in], outputs=out, name="GCN_Epsilon_Cosine")

gcn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR_GCN), loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH), weighted_metrics=["accuracy"])
gcn.summary()

# ============================================================
# STEP 5A — CALLBACKS
# ============================================================

ckpt_gcn = "best_GCN_Epsilon_Cosine.keras"
callbacks_gcn = [EarlyStopping(monitor="val_loss", patience=20, mode="min", restore_best_weights=True, verbose=1), ModelCheckpoint(ckpt_gcn, monitor="val_loss", mode="min", save_best_only=True, verbose=1)]

# ============================================================
# STEP 5B — TRAIN MODEL
# ============================================================

print("\n[STEP 5] Training GCN_Epsilon_Cosine...")
t4 = time.time()

hist = gcn.fit(x=[X_std, A_norm], y=y_onehot, sample_weight=sample_w_train, batch_size=N, epochs=EPOCHS_GCN, shuffle=False, verbose=1, validation_data=([X_std, A_norm], y_onehot, mask_va.astype(np.float32)), callbacks=callbacks_gcn)

t5 = time.time()
training_time = t5 - t4

print(f"\n[TIME] GCN training: {training_time / 60:.2f} minutes")

# ============================================================
# STEP 5C — TRAINING AND VALIDATION ACCURACY
# ============================================================

plt.figure(figsize=(7, 5))
plt.plot(hist.history["accuracy"], label="Training Accuracy")
plt.plot(hist.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("GCN ε-Cosine: Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# STEP 5D — TRAINING AND VALIDATION LOSS
# ============================================================

plt.figure(figsize=(7, 5))
plt.plot(hist.history["loss"], label="Training Loss")
plt.plot(hist.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GCN ε-Cosine: Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# STEP 6 — PREDICTIONS
# ============================================================

print("\n[STEP 6] Generating predictions...")
t6 = time.time()
y_prob = gcn.predict([X_std, A_norm], batch_size=N, verbose=0)
t7 = time.time()

inference_time = t7 - t6
y_pred = np.argmax(y_prob, axis=1)

print(f"[TIME] Full-graph inference: {inference_time:.4f} seconds")

# ============================================================
# STEP 6A — EVALUATION FUNCTION
# ============================================================

def evaluate_mask(mask_name, mask_bool):
    loss, acc = gcn.evaluate([X_std, A_norm], y_onehot, sample_weight=mask_bool.astype(np.float32), batch_size=N, verbose=0)

    print(f"\n{'=' * 60}")
    print(f"{mask_name} RESULTS")
    print(f"{'=' * 60}")
    print(f"Loss     : {loss:.4f}")
    print(f"Accuracy : {acc:.4f}")

    y_true_mask = labels_int[mask_bool]
    y_pred_mask = y_pred[mask_bool]
    y_prob_mask = y_prob[mask_bool]

    print(f"\n[{mask_name}] Classification Report:")
    print(classification_report(y_true_mask, y_pred_mask, labels=np.arange(num_classes), target_names=class_names, digits=4, zero_division=0))

    cm = confusion_matrix(y_true_mask, y_pred_mask, labels=np.arange(num_classes))
    print(f"[{mask_name}] Confusion Matrix:")
    print(cm)

    plt.figure(figsize=(6.5, 6))
    plt.imshow(cm, cmap="Blues")
    plt.colorbar()
    plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
    plt.yticks(range(num_classes), class_names)
    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.title(f"GCN ε-Cosine Confusion Matrix ({mask_name})")
    threshold = cm.max() / 2.0 if cm.size > 0 else 0

    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, cm[i, j], ha="center", va="center", color=("white" if cm[i, j] > threshold else "black"))

    plt.tight_layout()
    plt.show()

    try:
        y_bin = label_binarize(y_true_mask, classes=np.arange(num_classes))
        if num_classes == 2: y_bin = np.column_stack([1 - y_bin[:, 0], y_bin[:, 0]])

        fpr = {}
        tpr = {}
        roc_auc = {}

        for i in range(num_classes):
            if len(np.unique(y_bin[:, i])) < 2: continue
            fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_prob_mask[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_prob_mask.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
        valid_classes = [i for i in range(num_classes) if i in fpr]

        if len(valid_classes) > 0:
            all_fpr = np.unique(np.concatenate([fpr[i] for i in valid_classes]))
            mean_tpr = np.zeros_like(all_fpr)
            for i in valid_classes: mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
            mean_tpr /= len(valid_classes)
            fpr["macro"] = all_fpr
            tpr["macro"] = mean_tpr
            roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

        plt.figure(figsize=(8, 6))

        for i in valid_classes:
            plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.3f})")

        plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"Micro-average (AUC={roc_auc['micro']:.3f})")

        if "macro" in fpr:
            plt.plot(fpr["macro"], tpr["macro"], linestyle=":", label=f"Macro-average (AUC={roc_auc['macro']:.3f})")

        plt.plot([0, 1], [0, 1], "k--")
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"GCN ε-Cosine ROC Curves ({mask_name})")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.show()

        print(f"\n[{mask_name}] AUC:")
        for i in valid_classes: print(f"{class_names[i]}: {roc_auc[i]:.4f}")
        print(f"Micro-average: {roc_auc['micro']:.4f}")
        if "macro" in roc_auc: print(f"Macro-average: {roc_auc['macro']:.4f}")

    except Exception as e:
        print(f"[ROC] Skipped: {e}")

# ============================================================
# STEP 6B — VALIDATION AND TEST EVALUATION
# ============================================================

evaluate_mask("VALIDATION", mask_va)
evaluate_mask("TEST", mask_te)

# ============================================================
# FINAL COMPUTATIONAL INFORMATION — GCN_Epsilon_Cosine
# ============================================================

print("\n============================================================")
print("GCN_Epsilon_Cosine FINAL SUMMARY")
print("============================================================")
print(f"Model name           : {gcn.name}")
print(f"Input image size     : {IMG_SIZE_RAW[0]} × {IMG_SIZE_RAW[1]} × 3")
print(f"Raw feature dim      : {F}")
print(f"Number of nodes      : {N}")
print(f"Training nodes       : {len(idx_tr)}")
print(f"Validation nodes     : {len(idx_va)}")
print(f"Test nodes           : {len(idx_te)}")
print("Dataset split        : 70% / 20% / 10%")
print(f"Cosine threshold ε   : {EPS_COS}")
print(f"Cosine radius        : {radius:.2f}")
print("Neighbour metric     : Cosine")
print("Edge weighting       : Cosine similarity")
print(f"Graph edges          : {num_edges}")
print(f"Average degree       : {avg_degree:.2f}")
print(f"Minimum degree       : {min_degree}")
print(f"Maximum degree       : {max_degree}")
print(f"Graph density        : {graph_density:.6f}")
print(f"Graph components     : {n_components}")
print(f"Largest component    : {largest_cc}")
print(f"Adjacency nnz        : {adjacency_nnz}")
print(f"Class balancing      : {USE_CLASS_BALANCING}")
print(f"GCN hidden units     : {HIDDEN}")
print(f"Dropout              : {DROPOUT}")
print(f"Learning rate        : {LR_GCN}")
print(f"Weight decay         : {WD}")
print(f"Label smoothing      : {LABEL_SMOOTH}")
print(f"Epochs               : {EPOCHS_GCN}")
print(f"Trainable parameters : {gcn.count_params():,}")
print(f"Training time        : {training_time:.2f} seconds")
print(f"Training time        : {training_time / 60:.2f} minutes")
print(f"Inference time       : {inference_time:.4f} seconds")
print("============================================================")


***
<a name='import Packages'>
    
# 0 <span style='color:blue'>|</span> COMPLETE MEMORY / CACHE / VARIABLE CLEANUP

In [ ]:
import os, gc, time, psutil

def get_user_variables():
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None: return {}
        excluded = {"In", "Out", "get_ipython", "exit", "quit"}
        return {name: value for name, value in ip.user_ns.items() if not name.startswith("_") and name not in excluded}
    except Exception:
        return {}
def get_ram_usage():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2
def print_variable_information():
    variables = get_user_variables()
    print(f"Variable count  : {len(variables)}")
    if variables: print("Variables       : " + ", ".join(sorted(variables.keys())))
def clear_matplotlib_cache():
    try:
        import matplotlib.pyplot as plt
        print("\n[1] Closing Matplotlib figures...")
        plt.close("all")
        print("    Matplotlib figures closed.")
    except ImportError:
        print("\n[1] Matplotlib not installed.")
    except Exception as e:
        print(f"\n[1] Matplotlib warning: {e}")
def clear_tensorflow_memory():
    try:
        import tensorflow as tf
        print("\n[2] Clearing TensorFlow / Keras...")
        tf.keras.backend.clear_session()
        print("    Keras session cleared.")
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            for i in range(len(gpus)):
                try:
                    info = tf.config.experimental.get_memory_info(f"GPU:{i}")
                    print(f"    GPU:{i} current: {info['current']/1024**2:.2f} MB")
                    print(f"    GPU:{i} peak   : {info['peak']/1024**2:.2f} MB")
                except Exception:
                    pass
                try: tf.config.experimental.reset_memory_stats(f"GPU:{i}")
                except Exception: pass
        print("    TensorFlow cleanup completed.")
    except ImportError:
        print("\n[2] TensorFlow not installed.")
    except Exception as e:
        print(f"\n[2] TensorFlow warning: {e}")
def force_garbage_collection(cycles=3):
    print("\n[3] Running garbage collection...")
    total = sum(gc.collect() for _ in range(cycles))
    print(f"    Garbage-collected objects: {total}")
def delete_model_variables():
    print("\n[4] Deleting user/model/data variables...")
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None:
            print("    Not running inside Jupyter.")
            return 0, []
        protected = {"os", "gc", "time", "psutil", "get_user_variables", "get_ram_usage", "print_variable_information", "clear_matplotlib_cache", "clear_tensorflow_memory", "force_garbage_collection", "delete_model_variables", "clear_cache_and_memory", "In", "Out", "get_ipython", "exit", "quit"}
        names = [name for name in list(ip.user_ns.keys()) if not name.startswith("_") and name not in protected]
        print(f"    Variables selected for deletion: {len(names)}")
        if names: print("    Deleting: " + ", ".join(sorted(names)))
        deleted = 0
        for name in names:
            try:
                del ip.user_ns[name]
                deleted += 1
            except Exception as e:
                print(f"    Could not delete {name}: {e}")
        print(f"    Successfully deleted: {deleted}")
        return deleted, names
    except Exception as e:
        print(f"    Variable cleanup warning: {e}")
        return 0, []
def clear_cache_and_memory():
    print("\n" + "=" * 70)
    print("COMPLETE MEMORY / CACHE / VARIABLE CLEANUP")
    print("=" * 70)
    before_ram = get_ram_usage()
    before_variables = get_user_variables()
    before_count = len(before_variables)
    print("\nBEFORE CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {before_ram:.2f} MB")
    print(f"Variable count  : {before_count}")
    if before_variables: print("Variables       : " + ", ".join(sorted(before_variables.keys())))
    clear_matplotlib_cache()
    clear_tensorflow_memory()
    force_garbage_collection()
    deleted_count, deleted_names = delete_model_variables()
    print("\n[5] Final garbage collection...")
    total = sum(gc.collect() for _ in range(5))
    print(f"    Garbage-collected objects: {total}")
    time.sleep(1)
    after_ram = get_ram_usage()
    after_variables = get_user_variables()
    after_count = len(after_variables)
    print("\nAFTER CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {after_ram:.2f} MB")
    print(f"Variable count  : {after_count}")
    if after_variables: print("Remaining       : " + ", ".join(sorted(after_variables.keys())))
    ram_difference = before_ram - after_ram
    print("\n" + "=" * 70)
    print("CLEANUP SUMMARY")
    print("=" * 70)
    print(f"Variables before : {before_count}")
    print(f"Variables after  : {after_count}")
    print(f"Variables deleted: {deleted_count}")
    print("-" * 70)
    print(f"RAM before       : {before_ram:.2f} MB")
    print(f"RAM after        : {after_ram:.2f} MB")
    if ram_difference >= 0: print(f"RAM released     : {ram_difference:.2f} MB")
    else: print(f"RAM difference   : +{abs(ram_difference):.2f} MB")
    print("=" * 70)
    print("CLEANUP COMPLETED")
    print("=" * 70)
clear_cache_and_memory()


***
<a name='import Packages'>
    
# 4 <span style='color:blue'>|</span> GCN_RBF on Pure GCN on RAW PIXELS

In [ ]:
import os, glob, time
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from scipy.sparse.csgraph import connected_components
import tensorflow as tf
from tensorflow.keras import layers, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# ============================================================
# STEP 0 — PATHS AND PARAMETERS
# ============================================================

data_dir = r"D:/GCN/Brain_Tumor/four_class"
IMG_SIZE_RAW = (112, 112)
SEED = 123
K_RBF = 12
SIGMA_MODE = "auto"
SIGMA_FIXED = 1.0
MIN_WEIGHT = 1e-6
LR_GCN = 5e-3
WD = 5e-4
EPOCHS_GCN = 250
HIDDEN = 64
DROPOUT = 0.3
LABEL_SMOOTH = 0.05
TEST_SIZE = 0.10
VAL_SPLIT = 2 / 9
USE_CLASS_BALANCING = True
TSNE_MAX_N = 2500

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# STEP 1 — INDEX IMAGES AND LABELS
# ============================================================

class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c: i for i, c in enumerate(class_names)}
paths = []
labels_int = []

for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels_int.append(class_to_idx[c])

paths = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)

if len(paths) == 0: raise ValueError(f"No images found in {data_dir}")
if num_classes < 2: raise ValueError("At least two classes are required.")

print(f"[STEP 1] Found {len(paths)} images across {num_classes} classes:")
print(class_names)

# ============================================================
# STEP 1A — STRATIFIED 70/20/10 SPLIT
# ============================================================

all_idx = np.arange(len(paths))
idx_trainval, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_trainval, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_trainval])

N = len(paths)
mask_tr = np.zeros(N, dtype=bool)
mask_va = np.zeros(N, dtype=bool)
mask_te = np.zeros(N, dtype=bool)
mask_tr[idx_tr] = True
mask_va[idx_va] = True
mask_te[idx_te] = True

if len(idx_tr) + len(idx_va) + len(idx_te) != N: raise ValueError("Dataset split size mismatch.")
if np.any(mask_tr & mask_va) or np.any(mask_tr & mask_te) or np.any(mask_va & mask_te): raise ValueError("Dataset split overlap detected.")

print("\n=== Dataset Split ===")
print(f"Training   : {len(idx_tr)} ({100 * len(idx_tr) / N:.2f}%)")
print(f"Validation : {len(idx_va)} ({100 * len(idx_va) / N:.2f}%)")
print(f"Testing    : {len(idx_te)} ({100 * len(idx_te) / N:.2f}%)")

print("\n=== Class Distribution ===")
for i, cname in enumerate(class_names):
    n_train = np.sum(labels_int[idx_tr] == i)
    n_val = np.sum(labels_int[idx_va] == i)
    n_test = np.sum(labels_int[idx_te] == i)
    print(f"{cname}: Train={n_train}, Validation={n_val}, Test={n_test}")

# ============================================================
# STEP 2 — LOAD RAW PIXELS
# ============================================================

def load_flat(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE_RAW, method=tf.image.ResizeMethod.BILINEAR)
    img.set_shape([IMG_SIZE_RAW[0], IMG_SIZE_RAW[1], 3])
    return tf.reshape(img, [-1])

def extract_raw_features(paths_np, batch=64):
    feats = []
    for i in range(0, len(paths_np), batch):
        pb = paths_np[i:i + batch]
        xb = tf.stack([load_flat(p) for p in pb], axis=0).numpy()
        feats.append(xb)
    return np.vstack(feats).astype(np.float32)

print("\n[STEP 2] Extracting raw-pixel features...")
t0 = time.time()
X_raw = extract_raw_features(paths)
t1 = time.time()

print(f"[TIME] Feature extraction: {(t1 - t0) / 60:.2f} minutes")

N, F = X_raw.shape

print("\n=== Raw Feature Information ===")
print(f"Nodes N = {N}")
print(f"Feature dimension F = {F}")
print(f"Image dimensions = {IMG_SIZE_RAW[0]} × {IMG_SIZE_RAW[1]} × 3")

y_onehot = to_categorical(labels_int, num_classes=num_classes).astype(np.float32)

# ============================================================
# STEP 2A — STANDARDIZATION
# ============================================================

scaler = StandardScaler(with_mean=True, with_std=True)
scaler.fit(X_raw[idx_tr])
X_std = scaler.transform(X_raw).astype(np.float32)

print("[STEP 2A] Raw-pixel features standardized using training-set statistics.")

# ============================================================
# STEP 3 — BUILD RBF GRAPH
# ============================================================

print("\n[STEP 3] Building RBF graph...")
t2 = time.time()

nbrs = NearestNeighbors(n_neighbors=K_RBF + 1, metric="euclidean", n_jobs=-1)
nbrs.fit(X_std)
distances, indices = nbrs.kneighbors(X_std, return_distance=True)

# ============================================================
# STEP 3A — ESTIMATE SIGMA
# ============================================================

if SIGMA_MODE == "auto":
    nonzero_distances = distances[:, 1:].reshape(-1)
    nonzero_distances = nonzero_distances[nonzero_distances > 0]
    sigma_rbf = float(np.median(nonzero_distances))
else:
    sigma_rbf = float(SIGMA_FIXED)

sigma_rbf = max(sigma_rbf, 1e-8)
print(f"[RBF] Sigma = {sigma_rbf:.6f}")

# ============================================================
# STEP 3B — RBF WEIGHTS
# ============================================================

rows = []
cols = []
data = []
denominator = 2.0 * sigma_rbf * sigma_rbf

for i in range(N):
    for j, distance in zip(indices[i], distances[i]):
        if i == j: continue
        weight = np.exp(-(float(distance) * float(distance)) / denominator)
        if weight < MIN_WEIGHT: continue
        rows.append(i)
        cols.append(j)
        data.append(weight)

# ============================================================
# STEP 3C — CREATE SPARSE ADJACENCY MATRIX
# ============================================================

A_dir = sp.coo_matrix((np.asarray(data, dtype=np.float32), (np.asarray(rows, dtype=np.int32), np.asarray(cols, dtype=np.int32))), shape=(N, N), dtype=np.float32)
A_rbf = A_dir.maximum(A_dir.T)
A_rbf = A_rbf.tolil()
A_rbf.setdiag(1.0)
A_rbf = A_rbf.tocsr()
A_norm = gcn_filter(A_rbf).astype(np.float32)

t3 = time.time()
print(f"[TIME] RBF graph + normalization: {(t3 - t2) / 60:.2f} minutes")

# ============================================================
# STEP 3D — GRAPH STATISTICS
# ============================================================

A_no_self = A_rbf.copy().tocsr()
A_no_self.setdiag(0)
A_no_self.eliminate_zeros()

degree_counts = np.diff(A_no_self.indptr)
num_edges = A_no_self.nnz // 2
avg_degree = degree_counts.mean()
min_degree = degree_counts.min()
max_degree = degree_counts.max()
graph_density = (2.0 * num_edges) / (N * (N - 1)) if N > 1 else 0.0
adjacency_nnz = A_rbf.nnz
n_components, component_labels = connected_components(A_no_self, directed=False, return_labels=True)
component_sizes = np.bincount(component_labels)
largest_cc = component_sizes.max()

print("\n=== GCN_RBF Graph Statistics ===")
print(f"Nodes          : {N}")
print(f"Edges          : {num_edges}")
print(f"RBF K          : {K_RBF}")
print(f"Average degree : {avg_degree:.2f}")
print(f"Minimum degree : {min_degree}")
print(f"Maximum degree : {max_degree}")
print(f"Density        : {graph_density:.6f}")
print(f"Components     : {n_components}")
print(f"LargestCC      : {largest_cc}")
print(f"nnz(A)         : {adjacency_nnz}")

# ============================================================
# STEP 3E — t-SNE OF RAW FEATURES
# ============================================================

print("\n[VIS] Running t-SNE on raw features...")

if N > TSNE_MAX_N:
    sel = np.random.RandomState(SEED).choice(N, TSNE_MAX_N, replace=False)
    X_for_tsne = X_std[sel]
    y_for_tsne = labels_int[sel]
else:
    X_for_tsne = X_std
    y_for_tsne = labels_int

tsne = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=SEED)
X_2d = tsne.fit_transform(X_for_tsne)

plt.figure(figsize=(7, 6))
for i, cname in enumerate(class_names):
    m = y_for_tsne == i
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=8, alpha=0.7, label=cname)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE of Raw-Pixel Features")
plt.legend(markerscale=3, frameon=False)
plt.tight_layout()
plt.show()

# ============================================================
# STEP 3F — RBF DEGREE DISTRIBUTION
# ============================================================

plt.figure(figsize=(7, 4))
plt.hist(degree_counts, bins=range(int(degree_counts.min()), int(degree_counts.max()) + 2), edgecolor="black", alpha=0.85)
plt.xlabel("Node Degree Excluding Self-loop")
plt.ylabel("Count")
plt.title("Degree Distribution of RBF Graph")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 3G — RBF ADJACENCY SNAPSHOT
# ============================================================

subN = min(150, N)
A_small = A_rbf[:subN, :subN].toarray()
plt.figure(figsize=(5.5, 5))
plt.imshow(A_small, cmap="Greys", interpolation="nearest")
plt.xlabel("Node Index")
plt.ylabel("Node Index")
plt.title(f"RBF Adjacency Snapshot ({subN}×{subN})")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 4 — CLASS BALANCING
# ============================================================

train_counts = np.bincount(labels_int[idx_tr], minlength=num_classes).astype(np.float32)
class_w = np.ones(num_classes, dtype=np.float32)
sample_w_train = mask_tr.astype(np.float32)

if USE_CLASS_BALANCING:
    class_w = train_counts.sum() / (num_classes * np.maximum(train_counts, 1.0))
    sample_w_train = sample_w_train * class_w[labels_int]

print("\n=== Training Class Weights ===")
for i, cname in enumerate(class_names):
    print(f"{cname}: Count={int(train_counts[i])}, Weight={class_w[i]:.4f}")

# ============================================================
# STEP 5 — PURE GCN MODEL
# ============================================================

X_in = Input(shape=(X_std.shape[1],), name="X_in")
A_in = Input(shape=(N,), sparse=True, name="A_in")
h1 = GCNConv(HIDDEN, activation=None, kernel_regularizer=regularizers.l2(WD), name="GCN_1")([X_in, A_in])
h1 = layers.BatchNormalization(name="BatchNorm")(h1)
h1 = layers.Activation("relu", name="ReLU")(h1)
h1 = layers.Dropout(DROPOUT, name="Dropout")(h1)
res = layers.Dense(HIDDEN, use_bias=False, name="Residual_Projection")(X_in)
h1 = layers.Add(name="Residual_Add")([h1, res])
out = GCNConv(num_classes, activation="softmax", name="GCN_Output")([h1, A_in])
gcn = Model(inputs=[X_in, A_in], outputs=out, name="GCN_RBF")

gcn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR_GCN), loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH), weighted_metrics=["accuracy"])
gcn.summary()

# ============================================================
# STEP 5A — CALLBACKS
# ============================================================

ckpt_gcn = "best_GCN_RBF.keras"
callbacks_gcn = [EarlyStopping(monitor="val_loss", patience=20, mode="min", restore_best_weights=True, verbose=1), ModelCheckpoint(ckpt_gcn, monitor="val_loss", mode="min", save_best_only=True, verbose=1)]

# ============================================================
# STEP 5B — TRAIN GCN
# ============================================================

print("\n[STEP 5] Training GCN_RBF...")
t4 = time.time()

hist = gcn.fit(x=[X_std, A_norm], y=y_onehot, sample_weight=sample_w_train, batch_size=N, epochs=EPOCHS_GCN, shuffle=False, verbose=1, validation_data=([X_std, A_norm], y_onehot, mask_va.astype(np.float32)), callbacks=callbacks_gcn)

t5 = time.time()
training_time = t5 - t4

print(f"\n[TIME] GCN training: {training_time / 60:.2f} minutes")

# ============================================================
# STEP 5C — ACCURACY CURVES
# ============================================================

plt.figure(figsize=(7, 5))
plt.plot(hist.history["accuracy"], label="Training Accuracy")
plt.plot(hist.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("GCN RBF: Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# STEP 5D — LOSS CURVES
# ============================================================

plt.figure(figsize=(7, 5))
plt.plot(hist.history["loss"], label="Training Loss")
plt.plot(hist.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GCN RBF: Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# STEP 6 — PREDICTIONS
# ============================================================

print("\n[STEP 6] Generating predictions...")
t6 = time.time()
y_prob = gcn.predict([X_std, A_norm], batch_size=N, verbose=0)
t7 = time.time()

inference_time = t7 - t6
y_pred = np.argmax(y_prob, axis=1)

print(f"[TIME] Full-graph inference: {inference_time:.4f} seconds")

# ============================================================
# STEP 6A — EVALUATION
# ============================================================

def evaluate_mask(mask_name, mask_bool):
    loss, acc = gcn.evaluate([X_std, A_norm], y_onehot, sample_weight=mask_bool.astype(np.float32), batch_size=N, verbose=0)

    print(f"\n{'=' * 60}")
    print(f"{mask_name} RESULTS")
    print(f"{'=' * 60}")
    print(f"Loss     : {loss:.4f}")
    print(f"Accuracy : {acc:.4f}")

    y_true_mask = labels_int[mask_bool]
    y_pred_mask = y_pred[mask_bool]
    y_prob_mask = y_prob[mask_bool]

    print(f"\n[{mask_name}] Classification Report:")
    print(classification_report(y_true_mask, y_pred_mask, labels=np.arange(num_classes), target_names=class_names, digits=4, zero_division=0))

    cm = confusion_matrix(y_true_mask, y_pred_mask, labels=np.arange(num_classes))
    print(f"[{mask_name}] Confusion Matrix:")
    print(cm)

    plt.figure(figsize=(6.5, 6))
    plt.imshow(cm, cmap="Blues")
    plt.colorbar()
    plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
    plt.yticks(range(num_classes), class_names)
    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.title(f"GCN RBF Confusion Matrix ({mask_name})")
    threshold = cm.max() / 2.0 if cm.size > 0 else 0

    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, cm[i, j], ha="center", va="center", color=("white" if cm[i, j] > threshold else "black"))

    plt.tight_layout()
    plt.show()

    try:
        y_bin = label_binarize(y_true_mask, classes=np.arange(num_classes))
        if num_classes == 2: y_bin = np.column_stack([1 - y_bin[:, 0], y_bin[:, 0]])

        fpr = {}
        tpr = {}
        roc_auc = {}

        for i in range(num_classes):
            if len(np.unique(y_bin[:, i])) < 2: continue
            fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_prob_mask[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_prob_mask.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
        valid_classes = [i for i in range(num_classes) if i in fpr]

        if len(valid_classes) > 0:
            all_fpr = np.unique(np.concatenate([fpr[i] for i in valid_classes]))
            mean_tpr = np.zeros_like(all_fpr)
            for i in valid_classes: mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
            mean_tpr /= len(valid_classes)
            fpr["macro"] = all_fpr
            tpr["macro"] = mean_tpr
            roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

        plt.figure(figsize=(8, 6))

        for i in valid_classes:
            plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.3f})")

        plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"Micro-average (AUC={roc_auc['micro']:.3f})")

        if "macro" in fpr:
            plt.plot(fpr["macro"], tpr["macro"], linestyle=":", label=f"Macro-average (AUC={roc_auc['macro']:.3f})")

        plt.plot([0, 1], [0, 1], "k--")
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"GCN RBF ROC Curves ({mask_name})")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.show()

        print(f"\n[{mask_name}] AUC:")
        for i in valid_classes: print(f"{class_names[i]}: {roc_auc[i]:.4f}")
        print(f"Micro-average: {roc_auc['micro']:.4f}")
        if "macro" in roc_auc: print(f"Macro-average: {roc_auc['macro']:.4f}")

    except Exception as e:
        print(f"[ROC] Skipped: {e}")

# ============================================================
# STEP 6B — VALIDATION AND TEST
# ============================================================

evaluate_mask("VALIDATION", mask_va)
evaluate_mask("TEST", mask_te)

# ============================================================
# FINAL COMPUTATIONAL INFORMATION — GCN_RBF
# ============================================================

print("\n============================================================")
print("GCN_RBF FINAL SUMMARY")
print("============================================================")
print(f"Model name           : {gcn.name}")
print(f"Input image size     : {IMG_SIZE_RAW[0]} × {IMG_SIZE_RAW[1]} × 3")
print(f"Raw feature dim      : {F}")
print(f"Number of nodes      : {N}")
print(f"Training nodes       : {len(idx_tr)}")
print(f"Validation nodes     : {len(idx_va)}")
print(f"Test nodes           : {len(idx_te)}")
print("Dataset split        : 70% / 20% / 10%")
print(f"RBF K                : {K_RBF}")
print(f"RBF sigma            : {sigma_rbf:.6f}")
print(f"Sigma mode           : {SIGMA_MODE}")
print(f"Minimum edge weight  : {MIN_WEIGHT}")
print("Neighbour metric     : Euclidean")
print("Edge weighting       : RBF Gaussian similarity")
print(f"Graph edges          : {num_edges}")
print(f"Average degree       : {avg_degree:.2f}")
print(f"Minimum degree       : {min_degree}")
print(f"Maximum degree       : {max_degree}")
print(f"Graph density        : {graph_density:.6f}")
print(f"Graph components     : {n_components}")
print(f"Largest component    : {largest_cc}")
print(f"Adjacency nnz        : {adjacency_nnz}")
print(f"Class balancing      : {USE_CLASS_BALANCING}")
print(f"GCN hidden units     : {HIDDEN}")
print(f"Dropout              : {DROPOUT}")
print(f"Learning rate        : {LR_GCN}")
print(f"Weight decay         : {WD}")
print(f"Label smoothing      : {LABEL_SMOOTH}")
print(f"Epochs               : {EPOCHS_GCN}")
print(f"Trainable parameters : {gcn.count_params():,}")
print(f"Training time        : {training_time:.2f} seconds")
print(f"Training time        : {training_time / 60:.2f} minutes")
print(f"Inference time       : {inference_time:.4f} seconds")
print("============================================================")


***
<a name='import Packages'>
    
# 0 <span style='color:blue'>|</span> COMPLETE MEMORY / CACHE / VARIABLE CLEANUP

In [ ]:
# ============================================================
# COMPLETE MEMORY / CACHE / VARIABLE CLEANUP
# ============================================================

import os, gc, time, psutil
def get_user_variables():
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None: return {}
        excluded = {"In", "Out", "get_ipython", "exit", "quit"}
        return {name: value for name, value in ip.user_ns.items() if not name.startswith("_") and name not in excluded}
    except Exception:
        return {}
def get_ram_usage():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2
def print_variable_information():
    variables = get_user_variables()
    print(f"Variable count  : {len(variables)}")
    if variables: print("Variables       : " + ", ".join(sorted(variables.keys())))
def clear_matplotlib_cache():
    try:
        import matplotlib.pyplot as plt
        print("\n[1] Closing Matplotlib figures...")
        plt.close("all")
        print("    Matplotlib figures closed.")
    except ImportError:
        print("\n[1] Matplotlib not installed.")
    except Exception as e:
        print(f"\n[1] Matplotlib warning: {e}")
def clear_tensorflow_memory():
    try:
        import tensorflow as tf
        print("\n[2] Clearing TensorFlow / Keras...")
        tf.keras.backend.clear_session()
        print("    Keras session cleared.")
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            for i in range(len(gpus)):
                try:
                    info = tf.config.experimental.get_memory_info(f"GPU:{i}")
                    print(f"    GPU:{i} current: {info['current']/1024**2:.2f} MB")
                    print(f"    GPU:{i} peak   : {info['peak']/1024**2:.2f} MB")
                except Exception:
                    pass
                try: tf.config.experimental.reset_memory_stats(f"GPU:{i}")
                except Exception: pass
        print("    TensorFlow cleanup completed.")
    except ImportError:
        print("\n[2] TensorFlow not installed.")
    except Exception as e:
        print(f"\n[2] TensorFlow warning: {e}")
def force_garbage_collection(cycles=3):
    print("\n[3] Running garbage collection...")
    total = sum(gc.collect() for _ in range(cycles))
    print(f"    Garbage-collected objects: {total}")
def delete_model_variables():
    print("\n[4] Deleting user/model/data variables...")
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None:
            print("    Not running inside Jupyter.")
            return 0, []
        protected = {"os", "gc", "time", "psutil", "get_user_variables", "get_ram_usage", "print_variable_information", "clear_matplotlib_cache", "clear_tensorflow_memory", "force_garbage_collection", "delete_model_variables", "clear_cache_and_memory", "In", "Out", "get_ipython", "exit", "quit"}
        names = [name for name in list(ip.user_ns.keys()) if not name.startswith("_") and name not in protected]
        print(f"    Variables selected for deletion: {len(names)}")
        if names: print("    Deleting: " + ", ".join(sorted(names)))
        deleted = 0
        for name in names:
            try:
                del ip.user_ns[name]
                deleted += 1
            except Exception as e:
                print(f"    Could not delete {name}: {e}")
        print(f"    Successfully deleted: {deleted}")
        return deleted, names
    except Exception as e:
        print(f"    Variable cleanup warning: {e}")
        return 0, []
def clear_cache_and_memory():
    print("\n" + "=" * 70)
    print("COMPLETE MEMORY / CACHE / VARIABLE CLEANUP")
    print("=" * 70)
    before_ram = get_ram_usage()
    before_variables = get_user_variables()
    before_count = len(before_variables)
    print("\nBEFORE CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {before_ram:.2f} MB")
    print(f"Variable count  : {before_count}")
    if before_variables: print("Variables       : " + ", ".join(sorted(before_variables.keys())))
    clear_matplotlib_cache()
    clear_tensorflow_memory()
    force_garbage_collection()
    deleted_count, deleted_names = delete_model_variables()
    print("\n[5] Final garbage collection...")
    total = sum(gc.collect() for _ in range(5))
    print(f"    Garbage-collected objects: {total}")
    time.sleep(1)
    after_ram = get_ram_usage()
    after_variables = get_user_variables()
    after_count = len(after_variables)
    print("\nAFTER CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {after_ram:.2f} MB")
    print(f"Variable count  : {after_count}")
    if after_variables: print("Remaining       : " + ", ".join(sorted(after_variables.keys())))
    ram_difference = before_ram - after_ram
    print("\n" + "=" * 70)
    print("CLEANUP SUMMARY")
    print("=" * 70)
    print(f"Variables before : {before_count}")
    print(f"Variables after  : {after_count}")
    print(f"Variables deleted: {deleted_count}")
    print("-" * 70)
    print(f"RAM before       : {before_ram:.2f} MB")
    print(f"RAM after        : {after_ram:.2f} MB")
    if ram_difference >= 0: print(f"RAM released     : {ram_difference:.2f} MB")
    else: print(f"RAM difference   : +{abs(ram_difference):.2f} MB")
    print("=" * 70)
    print("CLEANUP COMPLETED")
    print("=" * 70)
clear_cache_and_memory()


***
<a name='import Packages'>
    
# 5 <span style='color:blue'>|</span> GCN on RAW PIXELS — Domain-Specific Graph

In [ ]:
import os, glob, time
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from scipy.sparse.csgraph import connected_components
import tensorflow as tf
from tensorflow.keras import layers, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter

# ============================================================
# STEP 0 — PATHS AND PARAMETERS
# ============================================================

data_dir = r"D:/GCN/Brain_Tumor/four_class"
IMG_SIZE_RAW = (112, 112)
SEED = 123
K_INTRA_TRAIN = 8
K_INTER_TRAIN = 2
K_UNLABELED = 12
BETA_INTRA = 1.2
ALPHA_INTER = 0.6
LR_GCN = 5e-3
WD = 5e-4
EPOCHS_GCN = 250
HIDDEN = 64
DROPOUT = 0.3
LABEL_SMOOTH = 0.05
TEST_SIZE = 0.10
VAL_SPLIT = 2 / 9
USE_CLASS_BALANCING = True
TSNE_MAX_N = 2500

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# STEP 1 — INDEX IMAGES AND LABELS
# ============================================================

class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c: i for i, c in enumerate(class_names)}
paths = []
labels_int = []

for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels_int.append(class_to_idx[c])

paths = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)

if len(paths) == 0: raise ValueError(f"No images found in {data_dir}")
if num_classes < 2: raise ValueError("At least two classes are required.")

print(f"[STEP 1] Found {len(paths)} images across {num_classes} classes:")
print(class_names)

# ============================================================
# STEP 1A — STRATIFIED 70/20/10 SPLIT
# ============================================================

all_idx = np.arange(len(paths))
idx_trainval, idx_te = train_test_split(all_idx, test_size=TEST_SIZE, random_state=SEED, stratify=labels_int)
idx_tr, idx_va = train_test_split(idx_trainval, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_int[idx_trainval])

N = len(paths)
mask_tr = np.zeros(N, dtype=bool)
mask_va = np.zeros(N, dtype=bool)
mask_te = np.zeros(N, dtype=bool)
mask_tr[idx_tr] = True
mask_va[idx_va] = True
mask_te[idx_te] = True

if len(idx_tr) + len(idx_va) + len(idx_te) != N: raise ValueError("Dataset split size mismatch.")
if np.any(mask_tr & mask_va) or np.any(mask_tr & mask_te) or np.any(mask_va & mask_te): raise ValueError("Dataset split overlap detected.")

print("\n=== Dataset Split ===")
print(f"Training   : {len(idx_tr)} ({100 * len(idx_tr) / N:.2f}%)")
print(f"Validation : {len(idx_va)} ({100 * len(idx_va) / N:.2f}%)")
print(f"Testing    : {len(idx_te)} ({100 * len(idx_te) / N:.2f}%)")

print("\n=== Class Distribution ===")
for i, cname in enumerate(class_names):
    n_train = np.sum(labels_int[idx_tr] == i)
    n_val = np.sum(labels_int[idx_va] == i)
    n_test = np.sum(labels_int[idx_te] == i)
    print(f"{cname}: Train={n_train}, Validation={n_val}, Test={n_test}")

# ============================================================
# STEP 2 — LOAD RAW PIXELS
# ============================================================

def load_flat(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE_RAW, method=tf.image.ResizeMethod.BILINEAR)
    img.set_shape([IMG_SIZE_RAW[0], IMG_SIZE_RAW[1], 3])
    return tf.reshape(img, [-1])

def extract_raw_features(paths_np, batch=64):
    feats = []
    for i in range(0, len(paths_np), batch):
        pb = paths_np[i:i + batch]
        xb = tf.stack([load_flat(p) for p in pb], axis=0).numpy()
        feats.append(xb)
    return np.vstack(feats).astype(np.float32)

print("\n[STEP 2] Extracting raw-pixel features...")
t0 = time.time()
X_raw = extract_raw_features(paths)
t1 = time.time()

print(f"[TIME] Feature extraction: {(t1 - t0) / 60:.2f} minutes")

N, F = X_raw.shape

print("\n=== Raw Feature Information ===")
print(f"Nodes N = {N}")
print(f"Feature dimension F = {F}")
print(f"Image dimensions = {IMG_SIZE_RAW[0]} × {IMG_SIZE_RAW[1]} × 3")

y_onehot = to_categorical(labels_int, num_classes=num_classes).astype(np.float32)

# ============================================================
# STEP 2A — STANDARDIZATION
# ============================================================

scaler = StandardScaler(with_mean=True, with_std=True)
scaler.fit(X_raw[idx_tr])
X_std = scaler.transform(X_raw).astype(np.float32)

print("[STEP 2A] Raw-pixel features standardized using training-set statistics.")

# ============================================================
# STEP 3 — BUILD DOMAIN-SPECIFIC GRAPH
# ============================================================

def build_domain_graph(X, labels, train_mask):
    print("\n[STEP 3] Building Domain-Specific Graph...")
    t2 = time.time()
    N_local = X.shape[0]
    idx_train = np.where(train_mask)[0]
    idx_unlabeled = np.where(~train_mask)[0]
    print("Training nodes:", len(idx_train))
    print("Unlabeled validation/test nodes:", len(idx_unlabeled))
    rows = []
    cols = []
    data = []

    print("[GRAPH] Adding same-class training edges...")
    for c in range(num_classes):
        idx_class = np.where((labels == c) & train_mask)[0]
        if len(idx_class) <= 1: continue
        n_neighbors = min(K_INTRA_TRAIN + 1, len(idx_class))
        nn_class = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", n_jobs=-1)
        nn_class.fit(X[idx_class])
        distances, neighbors = nn_class.kneighbors(X[idx_class], return_distance=True)
        for local_i in range(len(idx_class)):
            i = idx_class[local_i]
            for local_j, d in zip(neighbors[local_i], distances[local_i]):
                j = idx_class[local_j]
                if i == j: continue
                similarity = 1.0 - float(d)
                if similarity <= 0: continue
                weight = similarity * BETA_INTRA
                rows.append(i)
                cols.append(j)
                data.append(weight)

    print("[GRAPH] Adding cross-class training edges...")
    if len(idx_train) > 1 and K_INTER_TRAIN > 0:
        search_k = min(max(K_INTRA_TRAIN + K_INTER_TRAIN + 10, K_INTER_TRAIN + 1), len(idx_train))
        nn_train = NearestNeighbors(n_neighbors=search_k, metric="cosine", n_jobs=-1)
        nn_train.fit(X[idx_train])
        distances, neighbors = nn_train.kneighbors(X[idx_train], return_distance=True)
        for local_i in range(len(idx_train)):
            i = idx_train[local_i]
            added = 0
            for local_j, d in zip(neighbors[local_i], distances[local_i]):
                j = idx_train[local_j]
                if i == j: continue
                if labels[i] == labels[j]: continue
                similarity = 1.0 - float(d)
                if similarity <= 0: continue
                weight = similarity * ALPHA_INTER
                rows.append(i)
                cols.append(j)
                data.append(weight)
                added += 1
                if added >= K_INTER_TRAIN: break

    print("[GRAPH] Adding label-agnostic validation/test edges...")
    if len(idx_unlabeled) > 0 and K_UNLABELED > 0:
        n_neighbors = min(K_UNLABELED + 1, N_local)
        nn_global = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", n_jobs=-1)
        nn_global.fit(X)
        distances, neighbors = nn_global.kneighbors(X[idx_unlabeled], return_distance=True)
        for local_i in range(len(idx_unlabeled)):
            i = idx_unlabeled[local_i]
            added = 0
            for j, d in zip(neighbors[local_i], distances[local_i]):
                if i == j: continue
                similarity = 1.0 - float(d)
                if similarity <= 0: continue
                rows.append(i)
                cols.append(j)
                data.append(similarity)
                added += 1
                if added >= K_UNLABELED: break

    A_directed = sp.coo_matrix((np.asarray(data, dtype=np.float32), (np.asarray(rows, dtype=np.int32), np.asarray(cols, dtype=np.int32))), shape=(N_local, N_local), dtype=np.float32)
    A_domain = A_directed.maximum(A_directed.T)
    A_domain = A_domain.tolil()
    A_domain.setdiag(1.0)
    A_domain = A_domain.tocsr()
    A_normalized = gcn_filter(A_domain).astype(np.float32)
    t3 = time.time()
    print(f"[TIME] Domain graph + normalization: {(t3 - t2) / 60:.2f} minutes")
    return A_domain, A_normalized

A_domain, A_norm = build_domain_graph(X_std, labels_int, mask_tr)

# ============================================================
# STEP 3A — GRAPH STATISTICS
# ============================================================

A_no_self = A_domain.copy().tocsr()
A_no_self.setdiag(0)
A_no_self.eliminate_zeros()

degree_counts = np.diff(A_no_self.indptr)
num_edges = A_no_self.nnz // 2
avg_degree = degree_counts.mean()
min_degree = degree_counts.min()
max_degree = degree_counts.max()
graph_density = (2.0 * num_edges) / (N * (N - 1)) if N > 1 else 0.0
adjacency_nnz = A_domain.nnz
n_components, component_labels = connected_components(A_no_self, directed=False, return_labels=True)
component_sizes = np.bincount(component_labels)
largest_cc = component_sizes.max()

print("\n=== GCN_Domain_Specific Graph Statistics ===")
print(f"Nodes          : {N}")
print(f"Edges          : {num_edges}")
print(f"Average degree : {avg_degree:.2f}")
print(f"Minimum degree : {min_degree}")
print(f"Maximum degree : {max_degree}")
print(f"Density        : {graph_density:.6f}")
print(f"Components     : {n_components}")
print(f"LargestCC      : {largest_cc}")
print(f"nnz(A)         : {adjacency_nnz}")

# ============================================================
# STEP 3B — t-SNE OF RAW-PIXEL FEATURES
# ============================================================

print("\n[VIS] Running t-SNE on raw features...")

if N > TSNE_MAX_N:
    sel = np.random.RandomState(SEED).choice(N, TSNE_MAX_N, replace=False)
    X_for_tsne = X_std[sel]
    y_for_tsne = labels_int[sel]
else:
    X_for_tsne = X_std
    y_for_tsne = labels_int

tsne = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=SEED)
X_2d = tsne.fit_transform(X_for_tsne)

plt.figure(figsize=(7, 6))
for i, cname in enumerate(class_names):
    m = y_for_tsne == i
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=8, alpha=0.7, label=cname)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE of Raw-Pixel Features")
plt.legend(markerscale=3, frameon=False)
plt.tight_layout()
plt.show()

# ============================================================
# STEP 3C — DEGREE DISTRIBUTION
# ============================================================

plt.figure(figsize=(7, 4))
plt.hist(degree_counts, bins=range(int(degree_counts.min()), int(degree_counts.max()) + 2), edgecolor="black", alpha=0.85)
plt.xlabel("Node Degree Excluding Self-loop")
plt.ylabel("Count")
plt.title("Degree Distribution of Domain-Specific Graph")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 3D — ADJACENCY SNAPSHOT
# ============================================================

subN = min(150, N)
A_small = A_domain[:subN, :subN].toarray()
plt.figure(figsize=(5.5, 5))
plt.imshow(A_small, cmap="Greys", interpolation="nearest")
plt.xlabel("Node Index")
plt.ylabel("Node Index")
plt.title(f"Domain-Specific Adjacency Snapshot ({subN}×{subN})")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 4 — CLASS BALANCING
# ============================================================

train_counts = np.bincount(labels_int[idx_tr], minlength=num_classes).astype(np.float32)
class_w = np.ones(num_classes, dtype=np.float32)
sample_w_train = mask_tr.astype(np.float32)

if USE_CLASS_BALANCING:
    class_w = train_counts.sum() / (num_classes * np.maximum(train_counts, 1.0))
    sample_w_train = sample_w_train * class_w[labels_int]

print("\n=== Training Class Weights ===")
for i, cname in enumerate(class_names):
    print(f"{cname}: Count={int(train_counts[i])}, Weight={class_w[i]:.4f}")

# ============================================================
# STEP 5 — PURE GCN MODEL
# ============================================================

X_in = Input(shape=(X_std.shape[1],), name="X_in")
A_in = Input(shape=(N,), sparse=True, name="A_in")
h1 = GCNConv(HIDDEN, activation=None, kernel_regularizer=regularizers.l2(WD), name="GCN_1")([X_in, A_in])
h1 = layers.BatchNormalization(name="BatchNorm")(h1)
h1 = layers.Activation("relu", name="ReLU")(h1)
h1 = layers.Dropout(DROPOUT, name="Dropout")(h1)
res = layers.Dense(HIDDEN, use_bias=False, name="Residual_Projection")(X_in)
h1 = layers.Add(name="Residual_Add")([h1, res])
out = GCNConv(num_classes, activation="softmax", name="GCN_Output")([h1, A_in])
gcn = Model(inputs=[X_in, A_in], outputs=out, name="GCN_Domain_Specific")

gcn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR_GCN), loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH), weighted_metrics=["accuracy"])
gcn.summary()

# ============================================================
# STEP 5A — CALLBACKS
# ============================================================

ckpt_gcn = "best_GCN_Domain_Specific.keras"
callbacks_gcn = [EarlyStopping(monitor="val_loss", patience=20, mode="min", restore_best_weights=True, verbose=1), ModelCheckpoint(ckpt_gcn, monitor="val_loss", mode="min", save_best_only=True, verbose=1)]

# ============================================================
# STEP 5B — TRAIN GCN
# ============================================================

print("\n[STEP 5] Training GCN_Domain_Specific...")
t4 = time.time()

hist = gcn.fit(x=[X_std, A_norm], y=y_onehot, sample_weight=sample_w_train, batch_size=N, epochs=EPOCHS_GCN, shuffle=False, verbose=1, validation_data=([X_std, A_norm], y_onehot, mask_va.astype(np.float32)), callbacks=callbacks_gcn)

t5 = time.time()
training_time = t5 - t4

print(f"\n[TIME] GCN training: {training_time / 60:.2f} minutes")

# ============================================================
# STEP 5C — ACCURACY CURVES
# ============================================================

plt.figure(figsize=(7, 5))
plt.plot(hist.history["accuracy"], label="Training Accuracy")
plt.plot(hist.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("GCN Domain-Specific: Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# STEP 5D — LOSS CURVES
# ============================================================

plt.figure(figsize=(7, 5))
plt.plot(hist.history["loss"], label="Training Loss")
plt.plot(hist.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GCN Domain-Specific: Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# STEP 6 — PREDICTIONS
# ============================================================

print("\n[STEP 6] Generating predictions...")
t6 = time.time()
y_prob = gcn.predict([X_std, A_norm], batch_size=N, verbose=0)
t7 = time.time()

inference_time = t7 - t6
y_pred = np.argmax(y_prob, axis=1)

print(f"[TIME] Full-graph inference: {inference_time:.4f} seconds")

# ============================================================
# STEP 6A — EVALUATION
# ============================================================

def evaluate_mask(mask_name, mask_bool):
    loss, acc = gcn.evaluate([X_std, A_norm], y_onehot, sample_weight=mask_bool.astype(np.float32), batch_size=N, verbose=0)

    print(f"\n{'=' * 60}")
    print(f"{mask_name} RESULTS")
    print(f"{'=' * 60}")
    print(f"Loss     : {loss:.4f}")
    print(f"Accuracy : {acc:.4f}")

    y_true_mask = labels_int[mask_bool]
    y_pred_mask = y_pred[mask_bool]
    y_prob_mask = y_prob[mask_bool]

    print(f"\n[{mask_name}] Classification Report:")
    print(classification_report(y_true_mask, y_pred_mask, labels=np.arange(num_classes), target_names=class_names, digits=4, zero_division=0))

    cm = confusion_matrix(y_true_mask, y_pred_mask, labels=np.arange(num_classes))

    print(f"[{mask_name}] Confusion Matrix:")
    print(cm)

    plt.figure(figsize=(6.5, 6))
    plt.imshow(cm, cmap="Blues")
    plt.colorbar()
    plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
    plt.yticks(range(num_classes), class_names)
    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.title(f"GCN Domain-Specific Confusion Matrix ({mask_name})")
    threshold = cm.max() / 2.0 if cm.size > 0 else 0

    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, cm[i, j], ha="center", va="center", color=("white" if cm[i, j] > threshold else "black"))

    plt.tight_layout()
    plt.show()

    try:
        y_bin = label_binarize(y_true_mask, classes=np.arange(num_classes))
        if num_classes == 2: y_bin = np.column_stack([1 - y_bin[:, 0], y_bin[:, 0]])

        fpr = {}
        tpr = {}
        roc_auc = {}

        for i in range(num_classes):
            if len(np.unique(y_bin[:, i])) < 2: continue
            fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_prob_mask[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_prob_mask.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
        valid_classes = [i for i in range(num_classes) if i in fpr]

        if len(valid_classes) > 0:
            all_fpr = np.unique(np.concatenate([fpr[i] for i in valid_classes]))
            mean_tpr = np.zeros_like(all_fpr)
            for i in valid_classes: mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
            mean_tpr /= len(valid_classes)
            fpr["macro"] = all_fpr
            tpr["macro"] = mean_tpr
            roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

        plt.figure(figsize=(8, 6))

        for i in valid_classes:
            plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.3f})")

        plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"Micro-average (AUC={roc_auc['micro']:.3f})")

        if "macro" in fpr:
            plt.plot(fpr["macro"], tpr["macro"], linestyle=":", label=f"Macro-average (AUC={roc_auc['macro']:.3f})")

        plt.plot([0, 1], [0, 1], "k--")
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"GCN Domain-Specific ROC Curves ({mask_name})")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.show()

        print(f"\n[{mask_name}] AUC:")
        for i in valid_classes: print(f"{class_names[i]}: {roc_auc[i]:.4f}")
        print(f"Micro-average: {roc_auc['micro']:.4f}")
        if "macro" in roc_auc: print(f"Macro-average: {roc_auc['macro']:.4f}")

    except Exception as e:
        print(f"[ROC] Skipped: {e}")

# ============================================================
# STEP 6B — VALIDATION AND TEST
# ============================================================

evaluate_mask("VALIDATION", mask_va)
evaluate_mask("TEST", mask_te)

# ============================================================
# FINAL COMPUTATIONAL INFORMATION — GCN_Domain_Specific
# ============================================================

print("\n============================================================")
print("GCN_Domain_Specific FINAL SUMMARY")
print("============================================================")
print(f"Model name           : {gcn.name}")
print(f"Input image size     : {IMG_SIZE_RAW[0]} × {IMG_SIZE_RAW[1]} × 3")
print(f"Raw feature dim      : {F}")
print(f"Number of nodes      : {N}")
print(f"Training nodes       : {len(idx_tr)}")
print(f"Validation nodes     : {len(idx_va)}")
print(f"Test nodes           : {len(idx_te)}")
print("Dataset split        : 70% / 20% / 10%")
print(f"Intra-class K        : {K_INTRA_TRAIN}")
print(f"Inter-class K        : {K_INTER_TRAIN}")
print(f"Unlabeled K          : {K_UNLABELED}")
print(f"Intra-class weight   : {BETA_INTRA}")
print(f"Inter-class weight   : {ALPHA_INTER}")
print("Neighbour metric     : Cosine")
print("Graph construction   : Domain-specific label-aware training graph")
print(f"Graph edges          : {num_edges}")
print(f"Average degree       : {avg_degree:.2f}")
print(f"Minimum degree       : {min_degree}")
print(f"Maximum degree       : {max_degree}")
print(f"Graph density        : {graph_density:.6f}")
print(f"Graph components     : {n_components}")
print(f"Largest component    : {largest_cc}")
print(f"Adjacency nnz        : {adjacency_nnz}")
print(f"Class balancing      : {USE_CLASS_BALANCING}")
print(f"GCN hidden units     : {HIDDEN}")
print(f"Dropout              : {DROPOUT}")
print(f"Learning rate        : {LR_GCN}")
print(f"Weight decay         : {WD}")
print(f"Label smoothing      : {LABEL_SMOOTH}")
print(f"Epochs               : {EPOCHS_GCN}")
print(f"Trainable parameters : {gcn.count_params():,}")
print(f"Training time        : {training_time:.2f} seconds")
print(f"Training time        : {training_time / 60:.2f} minutes")
print(f"Inference time       : {inference_time:.4f} seconds")
print("============================================================")